# THE DATASET FOR PRETRAINING BERT

To pretrain the BERT model, we need to generate the dataset in the ideal format to facilitate the two pretraining tasks: masked language modeling and next sentence prediction. To facilitate the demonstration of BERT pretraining, we use a smaller corpus WikiText-2

Comparing with the PTB dataset used for pretraining word2vec, WikiText-2:
1. Retains the original punctuation, making it suitable for next sentence prediction
2. Retains the original case and numbers
3. Is over twice larger.

In [1]:
import torch
from utils import spy
import random 
import os

In the WikiText-2 dataset, each line represents a paragraph where space is inserted between any punctuation and its preceding token. Paragraphs with at least two sentences are retained. To split sentences, we only use the period as the delimiter for simplicity.

In [2]:
def _read_wiki(data_dir):
    file_name = os.path.join(data_dir, 'wikitext-2-v1-train.txt')
    with open(file_name, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Uppercase letters are converted to lowercase
    paragraphs = [line.strip().lower().split(' . ') 
                  for line in lines if len(line.split(' . ')) >= 2]
    random.shuffle(paragraphs)
    # paragraphs is a list of list of lists
    return paragraphs

## STEP 1 - Define helper functions for pretraining tasks

In the following, we begin by implementing helper functions for the two BERT pretraining tasks: next sentence prediction and masked language modeling. These helper functions will be invoked later when transforming the raw text corpus into the dataset of the ideal format to pretrain BERT.



### 1.1. Generating the Next Sequence Prediction task

The `_get_next_sentence` function generates a training example for the binary classification task.

In [3]:
def _get_next_sentence(sentence, next_sentence, paragraphs):
    if random.random() < 0.5:
        is_next = True
    else:
        next_sentence = random.choice(random.choice(paragraphs))
        is_next = False
    return sentence, next_sentence, is_next

The following function generates training examples for next sentence prediction from the input paragraph by invoking the `_get_next_sentence` function. Here `paragraph` is a list of sentences, where each sentence is a list of tokens. The argument `max_len` specifies the maximum length of a BERT input sequence during pretraining.

In [4]:
def _get_nsp_data_from_paragraph(paragraph, paragraphs, max_len):
    nsp_data_from_paragraph = []
    for i in range(len(paragraph) - 1):
        tokens_a, tokens_b, is_next = _get_next_sentence(
            paragraph[i], paragraph[i + 1], paragraphs
        )
        # 2 <sep> tokens and 1 <cls> token
        if len(tokens_a) + len(tokens_b) + 3 > max_len:
            continue
        tokens, segments = spy.get_tokens_and_segments(tokens_a, tokens_b)
        nsp_data_from_paragraph.append((tokens, segments, is_next))
    return nsp_data_from_paragraph

### 1.2. Generating the Masked Language Modeling task

In order to generate training examples for the masked language modeling task from a BERT input sequence, we define the following `_replace_mlm_tokens` function. In its inputs, tokens is a list of tokens representing a BERT input sequence, `candidate_pred_positions` is a list of token indices of the BERT input sequence excluding those of special tokens (special tokens are not predicted in the masked language modeling task), and `num_mlm_preds` indicates the number of predictions (recall 15% random tokens to predict). Following the definition of the masked language modeling task, at each prediction position, the input may be replaced by a special `<mask>` token or a random token, or remain unchanged. In the end, the function returns the input tokens after possible replacement, the token indices where predictions take place and labels for these predictions.

In [5]:
def _replace_mlm_tokens(tokens, candidate_pred_positions, num_mlm_preds, vocab):
    mlm_input_tokens = [token for token in tokens]
    pred_positions_and_labels = []
    random.shuffle(candidate_pred_positions)

    for mlm_pred_position in candidate_pred_positions:
        if len(pred_positions_and_labels) >= num_mlm_preds:
            break
        masked_token = None
        # 80% of the time -> replace the word with the '<mask>' token
        if random.random() < 0.8:
            masked_token = '<mask>'
        else:
            # 10% of the time -> keep the word unchanged
            if random.random() < 0.5:
                masked_token = tokens[mlm_pred_position]
            # 10% of the time -> replace the word with a random word
            else:
                masked_token = random.choice(vocab.idx_to_token)
        mlm_input_tokens[mlm_pred_position] = masked_token
        pred_positions_and_labels.append((mlm_pred_position, tokens[mlm_pred_position]))
    
    return mlm_input_tokens, pred_positions_and_labels

By invoking the aforementioned `_replace_mlm_tokens` function, the following function takes a BERT input sequence (tokens) as an input and returns indices of the input tokens (after possible token replacement), the token indices where predictions take place, and label indices for these predictions.

In [6]:
def _get_mlm_data_from_tokens(tokens, vocab):
    candidate_pred_positions = []
    # 'tokens' is a list of string
    for i, token in enumerate(tokens):
        if token in ['<cls>', '<sep>']:
            continue
        candidate_pred_positions.append(i)
    # 15% of random tokens are predicted in the masked language modeling task
    mlm_num_preds = max(1, round(len(tokens) * .15))
    mlm_input_tokens, pred_positions_and_labels = _replace_mlm_tokens(
        tokens, candidate_pred_positions, mlm_num_preds, vocab)
    pred_positions_and_labels = sorted(pred_positions_and_labels, key=lambda x: x[0])
    pred_positions = [v[0] for v in pred_positions_and_labels]
    mlm_pred_labels = [v[1] for v in pred_positions_and_labels]

    return vocab[mlm_input_tokens], pred_positions, vocab[mlm_pred_labels]

## STEP 2 - Transforming texts into the pretraining dataset

Now we are almost ready to customize a `Dataset` class for pretraining BERT. Before that, we still need to define a helper function `_pad_bert_inputs` to append the special `<pad>` tokens to the inputs. Its argument examples contain the outputs from the helper functions `_get_nsp_data_from_paragraph` and `_get_mlm_data_from_tokens` for the two pretraining tasks.

In [7]:
def _pad_BERT_inputs(examples, max_len, vocab):
    max_num_mlm_preds = round(max_len * .15)
    all_token_ids, all_segments, valid_lens = [], [], []
    all_pred_positions, all_mlm_weights, all_mlm_labels = [], [], []
    nsp_labels = []
    
    for (
        token_ids, 
        pred_positions, 
        mlm_pred_label_ids, 
        segments, 
        is_next
    ) in examples:
        all_token_ids.append(torch.tensor(
            token_ids + [vocab['<pad>']] * (max_len - len(token_ids)), 
            dtype=torch.long))
        all_segments.append(torch.tensor(
            segments + [0] * (max_len - len(token_ids)),
            dtype=torch.long))
        valid_lens.append(torch.tensor(len(token_ids), dtype=torch.float32))
        all_pred_positions.append(torch.tensor(
            pred_positions + [0] * (max_num_mlm_preds - len(mlm_pred_label_ids)),
            dtype=torch.long
        )) 
        all_mlm_weights.append(torch.tensor(
            [1.0] * len(pred_positions) + [0.0] * (max_num_mlm_preds - len(pred_positions)),
            dtype=torch.float32
        ))
        all_mlm_labels.append(torch.tensor(
            mlm_pred_label_ids + [0] * (max_num_mlm_preds - len(mlm_pred_label_ids)),
            dtype=torch.long
        ))
        nsp_labels.append(torch.tensor(
            is_next, dtype=torch.long
        ))
    
    return (all_token_ids, all_segments, valid_lens, all_pred_positions, 
            all_mlm_weights, all_mlm_labels, nsp_labels)

Putting the helper functions for generating training examples of the two pretraining tasks, and the helper function for padding inputs together, we customize the following `_WikiTextDataset` class as the WikiText-2 dataset for pretraining BERT. By implementing the `__getitem__` function, we can arbitrarily access the pretraining (masked language modeling and next sentence prediction) examples generated from a pair of sentences from the WikiText-2 corpus.


The original BERT model uses WordPiece embeddings whose vocabulary size is 30000. The tokenization method of WordPiece is a slight modification of the original byte pair encoding algorithm. For simplicity, we use the `spy.tokenize` function for tokenization. Infrequent tokens that appear less than five times are filtered out.

In [12]:
class _WikiTextDataset(torch.utils.data.Dataset):
    def __init__(self, paragraphs, max_len):
        paragraphs = [spy.tokenize(paragraph) for paragraph in paragraphs]
        sentences = [sentence for paragraph in paragraphs for sentence in paragraph]
        self.vocab = spy.Vocab(sentences, min_freq=5, reserved_tokens=[
            '<pad>', '<sep>', '<mask>', '<cls>'])
        # Get data for the next sentence prediction task
        examples = []
        for paragraph in paragraphs:
            examples.extend(_get_nsp_data_from_paragraph(
                paragraph, paragraphs, max_len
            ))
        # Get data for the masked language modeling task
        examples = [(_get_mlm_data_from_tokens(tokens, self.vocab) + (segments, is_next))
                    for (tokens, segments, is_next) in examples]
        # Pad BERT inputs
        (self.all_token_ids, self.all_segments, self.valid_lens, self.all_pred_positions, 
         self.all_mlm_weights, self.all_mlm_labels, self.nsp_labels) = _pad_BERT_inputs(
             examples, max_len, self.vocab
         )
        

    def __getitem__(self, idx):
        return (self.all_token_ids[idx], self.all_segments[idx], self.valid_lens[idx], 
                self.all_pred_positions[idx], self.all_mlm_weights[idx], 
                self.all_mlm_labels[idx], self.nsp_labels[idx])
    

    def __len__(self):
        return len(self.all_token_ids)
     

By using the `_read_wiki` function and the `_WikiTextDataset` class, we define the following `load_data_wiki` to download the WikiText-2 dataset and generate pretraining examples from it.



In [9]:
!pip install datasets

In [10]:
from datasets import load_dataset

def load_data_wiki(batch_size, max_len):
    """Load the WikiText-2 dataset"""
    data_dir = "../../data/wikitext-2"
    os.makedirs(data_dir, exist_ok=True)
    train_ds = load_dataset("Salesforce/wikitext", "wikitext-2-v1", split="train")

    with open(os.path.join(data_dir, "wikitext-2-v1-train.txt"), "w", encoding="utf-8") as f:
        for line in train_ds["text"]:
            f.write(line + "\n")
    
    paragraphs = _read_wiki(data_dir)
    train_set = _WikiTextDataset(paragraphs, max_len)
    train_iter = torch.utils.data.DataLoader(train_set, batch_size,
                                        shuffle=True)
    return train_iter, train_set.vocab

c:\Users\DELL\.conda\envs\torch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setting the batch size to 512 and the maximum length of a BERT input sequence to be 128, we print out the shapes of a minibatch of BERT pretraining examples. Note that in each BERT input sequence, $19 (128 \times 0.15)$ positions are predicted for the masked language modeling task.

In [13]:
batch_size, max_len = 512, 128
train_iter, vocab = load_data_wiki(batch_size, max_len)

for (tokens_X, segments_X, valid_lens_x, pred_positions_X, mlm_weights_X,
      mlm_Y, nsp_y) in train_iter:
      print(tokens_X.shape, segments_X.shape, valid_lens_x.shape,
          pred_positions_X.shape, mlm_weights_X.shape, mlm_Y.shape,
          nsp_y.shape)
      break

torch.Size([512, 128]) torch.Size([512, 128]) torch.Size([512]) torch.Size([512, 19]) torch.Size([512, 19]) torch.Size([512, 19]) torch.Size([512])
